## Unsupervised Model Evaluation - Breast Cancer Dataset
This notebook demonstrates key concepts in evaluating unsupervised learning models using the breast cancer dataset from sklearn.

In [ ]:
# Import required libraries for data manipulation and machine learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import scikit-learn modules
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score, davies_bouldin_score

: 

In [ ]:
# Load the breast cancer dataset from sklearn
data = load_breast_cancer()
X = data.data
y_true = data.target
feature_names = data.feature_names

# Display dataset information
print("=" * 60)
print("BREAST CANCER DATASET")
print("=" * 60)
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {data.target_names}")
print(f"Distribution: {np.bincount(y_true)}")
print("-" * 60)

# Display first few rows of data for inspection
print("\nFirst 5 rows (first 5 features):")
print(pd.DataFrame(X[:5, :5], columns=feature_names[:5]).round(2))

In [ ]:
# Standardize features to have mean=0 and std=1
# This ensures all features contribute equally to distance calculations
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verify standardization worked correctly
print("Data standardized")
print(f"Mean: {X_scaled.mean():.6f}")
print(f"Std: {X_scaled.std():.6f}")

In [ ]:
# Create DataFrame for easier plotting and analysis
df = pd.DataFrame(X_scaled, columns=feature_names)
df['target'] = y_true
df['target_label'] = data.target_names[y_true]

# Visualize first two features
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter plot: first two features colored by class
scatter = axes[0].scatter(df.iloc[:, 0], df.iloc[:, 1], 
                          c=df['target'], cmap='viridis', alpha=0.7)
axes[0].set_xlabel(feature_names[0])
axes[0].set_ylabel(feature_names[1])
axes[0].set_title('First Two Features by Class')
axes[0].legend(*scatter.legend_elements(), title="Class")

# Box plot: feature distributions by class
df.boxplot(column=feature_names[:4], by='target_label', ax=axes[1])
axes[1].set_title('Feature Distributions by Class')

# Display the plots
plt.tight_layout()
plt.show()

In [ ]:
# PART 1: Demonstrate why accuracy is meaningless for clustering

print("=" * 60)
print("PART 1: WHY ACCURACY FAILS FOR CLUSTERING")
print("=" * 60)

# Cluster with K=2 (true number of classes)
km = KMeans(n_clusters=2, n_init=10, random_state=42)
km.fit(X_scaled)
pred = km.labels_

# Calculate naive accuracy - the WRONG approach
naive_acc = np.mean(pred == y_true)
print(f"Naive accuracy: {naive_acc:.3f}")

# Demonstrate that renaming clusters changes the score arbitrarily
print("\nRenaming clusters changes the score:")
rng = np.random.default_rng(42)
for i in range(3):
    mapping = rng.permutation(2)
    renamed = mapping[pred]
    print(f"  Permutation {mapping} -> accuracy = {np.mean(renamed == y_true):.3f}")

print("\nThe grouping never changed. Only the names did.")
print("Use Adjusted Rand Index (ARI) instead.")

# Calculate ARI - the correct metric for clustering evaluation
ari = adjusted_rand_score(y_true, pred)
print(f"\nAdjusted Rand Index (ARI): {ari:.3f}")
print("=" * 60)

In [ ]:
# PART 2: Use silhouette score to find optimal number of clusters

print("=" * 60)
print("PART 2: SILHOUETTE SCORE - FINDING OPTIMAL K")
print("=" * 60)

# Define range of K values to test
k_range = range(2, 8)
silhouette_scores = []
davies_scores = []
inertias = []

# Calculate metrics for each K
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    
    # Calculate evaluation metrics
    sil = silhouette_score(X_scaled, labels)
    davies = davies_bouldin_score(X_scaled, labels)
    
    silhouette_scores.append(sil)
    davies_scores.append(davies)
    inertias.append(km.inertia_)
    
    print(f"K={k}: Silhouette={sil:.4f}, Davies-Bouldin={davies:.4f}")

# Find best K based on silhouette score
best_k = k_range[np.argmax(silhouette_scores)]
print(f"\nBest K by Silhouette: {best_k}")
print(f"True number of classes: 2")
print("=" * 60)

In [ ]:
# PART 2: Visualize silhouette scores and inertia curve

# Create subplots for both metrics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Silhouette score plot
axes[0].plot(list(k_range), silhouette_scores, 'o-', color='blue', linewidth=2)
axes[0].axvline(2, color='red', linestyle='--', label='True K=2')
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Score vs K')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Inertia curve (elbow method)
axes[1].plot(list(k_range), inertias, 'o-', color='green', linewidth=2)
axes[1].axvline(2, color='red', linestyle='--', label='True K=2')
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Inertia')
axes[1].set_title('Inertia Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Display the plots
plt.tight_layout()
plt.show()

In [ ]:
# PART 3: Test cluster stability using random subsamples

print("=" * 60)
print("PART 3: STABILITY TEST")
print("=" * 60)

def stability_test(X, k, n_trials=10):
    """
    Test cluster stability using random subsamples.
    High stability indicates real structure, low stability indicates noise.
    """
    ari_scores = []
    for _ in range(n_trials):
        # Take two random samples (80% of data each)
        idx1 = np.random.choice(len(X), size=int(0.8*len(X)), replace=False)
        idx2 = np.random.choice(len(X), size=int(0.8*len(X)), replace=False)
        
        # Cluster each sample
        km1 = KMeans(n_clusters=k, n_init=10, random_state=None)
        km2 = KMeans(n_clusters=k, n_init=10, random_state=None)
        km1.fit(X[idx1])
        km2.fit(X[idx2])
        
        # Find common points between samples
        common = np.intersect1d(idx1, idx2)
        if len(common) > 1:
            # Compare clusterings on common points
            labels1 = km1.predict(X[common])
            labels2 = km2.predict(X[common])
            ari_scores.append(adjusted_rand_score(labels1, labels2))
    
    return np.mean(ari_scores) if ari_scores else 0.0

# Calculate stability for each K
stabilities = []
for k in k_range:
    stab = stability_test(X_scaled, k)
    stabilities.append(stab)
    print(f"K={k}: Stability = {stab:.4f}")

# Find best K based on stability
best_stab = k_range[np.argmax(stabilities)]
print(f"\nBest K by Stability: {best_stab}")
print("High stability means clusters are real patterns, not noise")
print("=" * 60)

In [ ]:
# PART 3: Visualize stability scores

# Create plot for stability scores
plt.figure(figsize=(8, 5))
plt.plot(list(k_range), stabilities, 'o-', color='purple', linewidth=2)
plt.axvline(2, color='red', linestyle='--', label='True K=2')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Stability Score')
plt.title('Stability: Do Clusters Reappear in Different Samples?')
plt.legend()
plt.grid(True, alpha=0.3)

# Display the plot
plt.show()

In [ ]:
# PART 4: Detect overfitting using Gaussian Mixture Models

print("=" * 60)
print("PART 4: OVERFITTING DETECTION WITH GMM")
print("=" * 60)

# Split data into train and test sets
X_train, X_test = train_test_split(X_scaled, test_size=0.3, random_state=42)

# Define components to test
components = [1, 2, 3, 4, 6, 8, 10]
train_scores = []
test_scores = []
bic_scores = []

print("Components | Train Score | Test Score | BIC")
print("-" * 50)

# Train GMM with different numbers of components
for n in components:
    gmm = GaussianMixture(n_components=n, random_state=42)
    gmm.fit(X_train)
    
    # Store scores
    train_scores.append(gmm.score(X_train))
    test_scores.append(gmm.score(X_test))
    bic_scores.append(gmm.bic(X_train))
    
    print(f"    {n}       |   {gmm.score(X_train):.2f}   |   {gmm.score(X_test):.2f}   |   {gmm.bic(X_train):.1f}")

# Find optimal components by different criteria
best_test = components[np.argmax(test_scores)]
best_bic = components[np.argmin(bic_scores)]

print(f"\nOptimal by test score: {best_test} components")
print(f"Optimal by BIC: {best_bic} components")
print(f"True number of classes: 2")
print("\nWhen test score drops while train score rises = overfitting")
print("=" * 60)

In [ ]:
# PART 4: Visualize GMM overfitting

# Create subplots for both metrics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training vs Test scores
axes[0].plot(components, train_scores, 'o-', color='blue', label='Train')
axes[0].plot(components, test_scores, 'o-', color='green', label='Test')
axes[0].axvline(2, color='red', linestyle='--', label='True K=2')
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Log-Likelihood')
axes[0].set_title('Training vs Test Score')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# BIC scores
axes[1].plot(components, bic_scores, 'o-', color='orange', linewidth=2)
axes[1].axvline(2, color='red', linestyle='--', label='True K=2')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('BIC (lower is better)')
axes[1].set_title('Bayesian Information Criterion')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Display the plots
plt.tight_layout()
plt.show()

In [ ]:
# PART 5: Explore DBSCAN's complexity dial (epsilon)

print("=" * 60)
print("PART 5: DBSCAN - THE COMPLEXITY DIAL")
print("=" * 60)

# Epsilon is the complexity dial for DBSCAN
eps_values = [0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
print("eps | Clusters | Noise % | Silhouette | Verdict")
print("-" * 55)

# Test different epsilon values
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_scaled)
    
    # Calculate metrics
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_pct = 100 * np.mean(labels == -1)
    sil = silhouette_score(X_scaled, labels) if n_clusters >= 2 else 0
    
    # Determine verdict for this epsilon value
    if n_clusters == 0:
        verdict = "All noise"
    elif n_clusters == 2 and noise_pct < 20:
        verdict = "OPTIMAL"
    elif noise_pct > 50:
        verdict = "Overfitting (eps too small)"
    elif n_clusters < 2:
        verdict = "Underfitting (eps too large)"
    else:
        verdict = "Suboptimal"
    
    print(f"{eps:.1f}  |    {n_clusters}     |   {noise_pct:.1f}%   |   {sil:.3f}   | {verdict}")

print("\nToo small eps -> overfitting (high noise, many clusters)")
print("Too large eps -> underfitting (merged clusters)")
print("Optimal eps finds correct clusters with low noise")
print("=" * 60)

In [ ]:
# PART 5: Visualize optimal DBSCAN clustering

# Use optimal epsilon from previous results
best_eps = 0.8
db_optimal = DBSCAN(eps=best_eps, min_samples=5)
labels_optimal = db_optimal.fit_predict(X_scaled)

# Plot first two features colored by DBSCAN labels
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], 
                      c=labels_optimal, cmap='viridis', alpha=0.7)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title(f'DBSCAN Clustering (eps={best_eps})')
plt.colorbar(scatter, label='Cluster')

# Display the plot
plt.show()

In [ ]:
# FINAL COMPARISON: K=2 vs K=3

print("=" * 60)
print("COMPARISON: K=2 vs K=3")
print("=" * 60)

# Run K-Means with K=2 and K=3
km2 = KMeans(n_clusters=2, n_init=10, random_state=42)
km3 = KMeans(n_clusters=3, n_init=10, random_state=42)
labels2 = km2.fit_predict(X_scaled)
labels3 = km3.fit_predict(X_scaled)

# Calculate and display all metrics
print(f"""
{'Metric':<20} {'K=2':<12} {'K=3':<12}
{'-'*44}
Silhouette        {silhouette_score(X_scaled, labels2):.4f}      {silhouette_score(X_scaled, labels3):.4f}
Davies-Bouldin    {davies_bouldin_score(X_scaled, labels2):.4f}      {davies_bouldin_score(X_scaled, labels3):.4f}
ARI               {adjusted_rand_score(y_true, labels2):.4f}      {adjusted_rand_score(y_true, labels3):.4f}
Stability         {stability_test(X_scaled, 2, n_trials=5):.4f}      {stability_test(X_scaled, 3, n_trials=5):.4f}
""")

# Draw conclusion
print("K=2 outperforms K=3 on all metrics")
print("This validates the clinical distinction: Malignant vs Benign")
print("=" * 60)